# Cosmic web — the U-Net step: recover the web from sparse tracers

The mapping notebook classified a density field we *fully* observe. A real galaxy survey doesn't get that — it sees a sparse, shot-noisy *sampling* of the matter field. So the real question is: **can we recover the cosmic-web class (void / sheet / filament / knot) from limited tracers?** That's what this notebook trains a U-Net to do.

Two honest simplifications to keep it laptop-friendly:
- **2D on slices.** The web is 3D, but a 2D U-Net on slices trains on a CPU in minutes. The 3D version is the *same network* with `Conv2d → Conv3d` (and it wants a GPU). We prove the concept in 2D first.
- **Small.** Tiny grid, small net, few epochs. Everything below runs without a GPU.

Pipeline: generate a Zel'dovich field → label its web with the tidal-tensor method (the "ground truth") → make a sparse-tracer view of the same field (the "input") → train the U-Net to map input → web class → score it against a majority-class baseline.

## 0. Setup

PyTorch (CPU) is all you need. On Windows, `pip install torch` gives the CPU build by default — no CUDA, no compiler.

In [ ]:
# run once
!pip install torch

In [ ]:
import os, time, numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

torch.manual_seed(0); np.random.seed(0)
torch.set_num_threads(os.cpu_count() or 1)   # use all laptop cores

# --- config ---
N       = 64       # grid per side (64 trains fast on CPU; 96/128 = sharper, slower)
L       = 250.0    # box, Mpc/h
nbar    = 2.0      # mean tracers per voxel: LOWER = sparser = harder
f       = 16       # U-Net width
epochs  = 12
CLASS_NAMES  = ['void', 'sheet', 'filament', 'knot']
CLASS_COLORS = ['#08306b', '#6baed6', '#fd8d3c', '#cb181d']
print("torch", torch.__version__, "| threads", torch.get_num_threads())

## 1. Data: a field, its web labels, and a sparse-tracer view

Same Zel'dovich generator and tidal-tensor classifier as the mapping notebook. The new piece is `make_cube`, which also draws Poisson tracers with rate proportional to (1+δ) — dense regions get more tracers, voids get almost none, exactly how galaxies sparsely trace matter. We build two independent realizations (different seeds) so train and validation never share a cube.

In [ ]:
def Pk_bbks(k, ns=0.96, G=0.2):
    k = np.asarray(k, float); q = np.where(k > 0, k/G, 1e-10)
    T = (np.log(1+2.34*q)/(2.34*q)) * (1+3.89*q+(16.1*q)**2+(5.46*q)**3+(6.71*q)**4)**(-0.25)
    return k**ns * T**2

def cic(pos, N, L):
    g = np.zeros((N, N, N)); s = (pos % L)/(L/N); i = np.floor(s).astype(int) % N; fr = s - np.floor(s)
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                wx = fr[:,0] if dx else 1-fr[:,0]; wy = fr[:,1] if dy else 1-fr[:,1]; wz = fr[:,2] if dz else 1-fr[:,2]
                ii=(i[:,0]+dx)%N; jj=(i[:,1]+dy)%N; kk=(i[:,2]+dz)%N
                np.add.at(g, (ii,jj,kk), wx*wy*wz)
    return g

def zeldovich(N, L, rms=8.0, seed=42):
    kax = np.fft.fftfreq(N, d=L/N)*2*np.pi; KX,KY,KZ = np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0
    rng = np.random.default_rng(seed)
    dk = np.fft.fftn(rng.standard_normal((N,N,N)))*np.sqrt(Pk_bbks(np.sqrt(k2))); dk[0,0,0]=0
    q = np.arange(N)*(L/N); QX,QY,QZ = np.meshgrid(q,q,q,indexing='ij'); Q=[QX,QY,QZ]
    disp = np.array([np.fft.ifftn(1j*Kv[i]*dk/k2).real for i in range(3)])
    disp *= rms/np.sqrt((disp**2).sum(0)).std()
    pos = np.stack([(Q[i]+disp[i]).ravel() for i in range(3)], axis=1)
    rho = cic(pos, N, L); return rho/rho.mean()-1.0

def smooth(d, L, R):
    N=d.shape[0]; kax=np.fft.fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return np.fft.ifftn(np.fft.fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real

def tweb(d, L, lam=0.3):
    N=d.shape[0]; kax=np.fft.fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=np.fft.fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=np.fft.ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T) > lam, -1).astype(np.int64)

def make_cube(N, L, seed, nbar):
    d   = zeldovich(N, L, 8.0, seed)
    lab = tweb(smooth(d, L, 3.0), L, 0.3)                 # ground-truth web from the FULL field
    rng = np.random.default_rng(seed + 999)
    counts = rng.poisson(nbar * np.clip(1 + d, 0, None))  # sparse tracers ~ (1+delta)
    dsp = counts / counts.mean() - 1.0                    # the INPUT view
    return dsp.astype(np.float32), lab

def to_slices(dsp, lab):
    X, Y = [], []
    for ax in range(3):                                   # slice along all 3 axes for 3x data
        ds = np.moveaxis(dsp, ax, 0); ls = np.moveaxis(lab, ax, 0)
        for k in range(ds.shape[0]):
            X.append(ds[k]); Y.append(ls[k])
    return np.array(X), np.array(Y)

Xtr, Ytr = to_slices(*make_cube(N, L, seed=1, nbar=nbar))
Xva, Yva = to_slices(*make_cube(N, L, seed=2, nbar=nbar))
freq = np.bincount(Ytr.ravel(), minlength=4)
print(f"train {Xtr.shape}  val {Xva.shape}")
print("class mix %:", (100*freq/freq.sum()).round(1), dict(zip(CLASS_NAMES, (100*freq/freq.sum()).round(1))))

## 2. The 2D U-Net

A small encoder-decoder: two downsampling blocks, a bottleneck, two upsampling blocks with skip connections, and a 1x1 conv to four class logits. To go 3D later, swap every `Conv2d`/`MaxPool2d`/`ConvTranspose2d` for its 3D counterpart — the structure is identical.

In [ ]:
def double_conv(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True))

class UNet2D(nn.Module):
    def __init__(self, f=16, n_classes=4):
        super().__init__()
        self.enc1 = double_conv(1, f)
        self.enc2 = double_conv(f, 2*f)
        self.bott = double_conv(2*f, 4*f)
        self.up2  = nn.ConvTranspose2d(4*f, 2*f, 2, 2); self.dec2 = double_conv(4*f, 2*f)
        self.up1  = nn.ConvTranspose2d(2*f, f, 2, 2);   self.dec1 = double_conv(2*f, f)
        self.out  = nn.Conv2d(f, n_classes, 1)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b  = self.bott(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(b),  e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

net = UNet2D(f)
print(net.__class__.__name__, "-", sum(p.numel() for p in net.parameters()), "parameters")

## 3. Train

Cross-entropy with inverse-frequency class weights (so the rare knot class isn't ignored), Adam, a dozen epochs. We print validation accuracy and per-class F1, and compare to the majority-class baseline (always predict 'void').

In [ ]:
mu, sd = Xtr.mean(), Xtr.std()                       # standardize the input
Xtr_t = torch.tensor((Xtr-mu)/sd)[:, None]; Ytr_t = torch.tensor(Ytr)
Xva_t = torch.tensor((Xva-mu)/sd)[:, None]; Yva_t = torch.tensor(Yva)

w = 1.0/(freq+1); w = (w/w.sum()*4).astype(np.float32)
crit = nn.CrossEntropyLoss(weight=torch.tensor(w))
opt  = torch.optim.Adam(net.parameters(), 1e-3)

def per_class_f1(pred, true, nc=4):
    out=[]
    for c in range(nc):
        tp=((pred==c)&(true==c)).sum(); fp=((pred==c)&(true!=c)).sum(); fn=((pred!=c)&(true==c)).sum()
        out.append(2*tp/(2*tp+fp+fn+1e-9))
    return out

bs, n = 16, len(Xtr_t); t0=time.time()
for ep in range(epochs):
    net.train(); perm = torch.randperm(n)
    for i in range(0, n, bs):
        idx = perm[i:i+bs]; opt.zero_grad()
        loss = crit(net(Xtr_t[idx]), Ytr_t[idx]); loss.backward(); opt.step()
    if ep % 3 == 2 or ep == epochs-1:
        net.eval()
        with torch.no_grad(): pv = net(Xva_t).argmax(1)
        acc = (pv == Yva_t).float().mean().item()
        f1 = per_class_f1(pv.numpy(), Yva_t.numpy())
        print(f"epoch {ep+1:2d}  loss {loss.item():.3f}  valAcc {acc:.3f}   "
              f"F1: void {f1[0]:.2f}  sheet {f1[1]:.2f}  filament {f1[2]:.2f}  knot {f1[3]:.2f}   ({time.time()-t0:.0f}s)")

maj = freq.argmax()
print(f"\nmajority-class baseline valAcc = {(Yva==maj).mean():.3f}  (always predict '{CLASS_NAMES[maj]}')")

## 4. See it work

Left: the sparse tracer field the network actually sees. Middle: the true web from the full field. Right: what the U-Net reconstructs from the sparse input alone.

In [ ]:
net.eval()
with torch.no_grad(): pv = net(Xva_t).argmax(1).numpy()
s = len(Xva)//2
cmap = ListedColormap(CLASS_COLORS); norm = BoundaryNorm([-.5,.5,1.5,2.5,3.5], 4)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(Xva[s].T, origin='lower', cmap='inferno'); ax[0].set_title(f'input: sparse tracers (nbar={nbar})')
ax[1].imshow(Yva[s].T, origin='lower', cmap=cmap, norm=norm); ax[1].set_title('true web (full field)')
im = ax[2].imshow(pv[s].T, origin='lower', cmap=cmap, norm=norm); ax[2].set_title('U-Net prediction')
cb = plt.colorbar(im, ax=ax[2], ticks=[0,1,2,3], fraction=0.046)
cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## Next steps

- **Make it harder.** Drop `nbar` (e.g. 1.0, 0.5) to mimic a sparser survey and watch where the net breaks down first — usually the thin filaments and rare knots. This is the honest measure of how much web information survives in limited tracers.
- **Go 3D.** Swap `Conv2d`/`MaxPool2d`/`ConvTranspose2d` for the 3D versions and train on sub-cubes instead of slices. Same code, more memory — this is the point to move to a GPU (Colab's free tier is enough).
- **2LPT upgrade (your next build).** Replace the Zel'dovich field with a second-order (2LPT) one for crisper filaments and knots, then retrain — a cleaner target makes the inference task sharper.
- **Real data, eventually.** Once a real QUIJOTE `df_m_*.npy` is in hand (Route A of the mapping notebook), label it the same way and test whether a net trained on simulations transfers to it.